## Window Function
* A window function calculates a return value for every input row of a DataFrame based on a specific group of rows, called a **frame**

It is distinct from a standard groupBy aggregation in the following way:
* **GroupBy:** Collapses all rows in a group into a single result row (e.g., calculating the total sales per country returns one row per country).
* **Window Function:** Maintains the original number of rows but appends a new column with a calculated value for each row based on its group (e.g., calculating the rank of a specific sale within a country, or a running total)

### Core Components of a Window Specification
1.  **Partitioning (`partitionBy`):** This controls how rows are grouped. It is conceptually similar to a `groupBy`, isolating calculations to specific groups (e.g., calculating ranks *per customer*).
2.  **Ordering (`orderBy`):** This determines the order of rows within a partition. Ordering is essential for functions like ranking or calculating cumulative totals.
3.  **Frame Specification (`rowsBetween` / `rangeBetween`):** This defines exactly which rows are included in the calculation for the current row (e.g., "all previous rows up to the current row" for a running total).

### Types of Window Functions
Spark supports three categories of window functions:
| Category | Functions | Description |
| :--- | :--- | :--- |
| **Ranking** | `rank`, `dense_rank`, `percent_rank`, `ntile`, `row_number` | Assigns a rank to rows within a partition based on the specified order. |
| **Analytic** | `cume_dist`, `first_value`, `last_value`, `lag`, `lead` | Accesses data from other rows in the window (e.g., `lag` gets the value from the previous row). |
| **Aggregate** | `sum`, `count`, `avg`, `min`, `max` | Performs standard aggregations over the defined window frame. |


In [82]:
import $ivy.`org.apache.spark:spark-core_2.12:3.5.3`
import $ivy.`org.apache.spark:spark-sql_2.12:3.5.3`
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.types._
import org.apache.spark.sql.expressions.Window

import $ivy.$                                       

import $ivy.$                                      

import org.apache.spark.sql.SparkSession

import org.apache.spark.sql.functions._

import org.apache.spark.sql.DataFrame

import org.apache.spark.sql.types._

import org.apache.spark.sql.expressions.Window

In [83]:
val spark = SparkSession.builder().master("local[*]").appName("readDF").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
import spark.implicits._

spark: SparkSession = org.apache.spark.sql.SparkSession@657e5ee1
import spark.implicits._

## creating df from array

In [84]:
// defining the data in Array
val productData: Array[(Int, String, Double, String, String)] = Array(
(1, "Laptop", 1200.00, "Electronics"    , "2026-02-25"),
(2, "Mouse", 25.00, "Electronics"       , "2026-02-25"),
(3, "Keyboard", 45.00, "Electronics"    , "2026-02-25"),
(4, "Monitor", 200.00, "Electronics"    , "2026-02-25"),
(5, "Desk Chair", 150.00, "Furniture"   , "2026-02-25"),
(6, "Desk Lamp", 30.00, "Furniture"     , "2026-02-26"),
(7, "Notebook", 5.00, "Stationery"      , "2026-02-26"),
(8, "Pen Set", 12.00, "Stationery"      , "2026-02-27"),
(9, "Charger", 200.00, "Electronics"    , "2026-02-27"),
(10, "Cell", 200.00, "Electronics"      , "2026-02-28")
)
val df = productData.toSeq.toDF("product_id", "product_name", "price", "category", "Date")


productData: Array[(Int, String, Double, String, String)] = Array(
  (1, "Laptop", 1200.0, "Electronics", "2026-02-25"),
  (2, "Mouse", 25.0, "Electronics", "2026-02-25"),
  (3, "Keyboard", 45.0, "Electronics", "2026-02-25"),
  (4, "Monitor", 200.0, "Electronics", "2026-02-25"),
  (5, "Desk Chair", 150.0, "Furniture", "2026-02-25"),
  (6, "Desk Lamp", 30.0, "Furniture", "2026-02-26"),
  (7, "Notebook", 5.0, "Stationery", "2026-02-26"),
  (8, "Pen Set", 12.0, "Stationery", "2026-02-27"),
  (9, "Charger", 200.0, "Electronics", "2026-02-27"),
  (10, "Cell", 200.0, "Electronics", "2026-02-28")
)
df: DataFrame = [product_id: int, product_name: string ... 3 more fields]

In [85]:
df.show()

+----------+------------+------+-----------+----------+
|product_id|product_name| price|   category|      Date|
+----------+------------+------+-----------+----------+
|         1|      Laptop|1200.0|Electronics|2026-02-25|
|         2|       Mouse|  25.0|Electronics|2026-02-25|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|
|         4|     Monitor| 200.0|Electronics|2026-02-25|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|
|         7|    Notebook|   5.0| Stationery|2026-02-26|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|
|         9|     Charger| 200.0|Electronics|2026-02-27|
|        10|        Cell| 200.0|Electronics|2026-02-28|
+----------+------------+------+-----------+----------+



## calculate the running sum of price order by price in asc order
option01: in this way we can calculate the running sum but in case of duplicate value of price, running sum will be same for that duplicate value.

In [86]:
// we are going to use the window function to do this
val WindowFun = Window.orderBy(col("price").asc)

// option01: implementing window function on df by creating column
df.withColumn("running_sum", sum(col("price")).over(WindowFun)).show()

+----------+------------+------+-----------+----------+-----------+
|product_id|product_name| price|   category|      Date|running_sum|
+----------+------------+------+-----------+----------+-----------+
|         7|    Notebook|   5.0| Stationery|2026-02-26|        5.0|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|       17.0|
|         2|       Mouse|  25.0|Electronics|2026-02-25|       42.0|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|       72.0|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|      117.0|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|      267.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|      867.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|      867.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|      867.0|
|         1|      Laptop|1200.0|Electronics|2026-02-25|     2067.0|
+----------+------------+------+-----------+----------+-----------+



WindowFun: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@6e4581a6

In [87]:
// option02: implementing on select statement
df.select(col("*"), sum(col("price")).over(Window.orderBy(col("price").asc)).alias("running_sum")).show()

+----------+------------+------+-----------+----------+-----------+
|product_id|product_name| price|   category|      Date|running_sum|
+----------+------------+------+-----------+----------+-----------+
|         7|    Notebook|   5.0| Stationery|2026-02-26|        5.0|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|       17.0|
|         2|       Mouse|  25.0|Electronics|2026-02-25|       42.0|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|       72.0|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|      117.0|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|      267.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|      867.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|      867.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|      867.0|
|         1|      Laptop|1200.0|Electronics|2026-02-25|     2067.0|
+----------+------------+------+-----------+----------+-----------+



**Option02**: if we will see the below running sum then product_id 4,9 and 10 having same price and running sum we are getting as same.
|product_id|product_name| price|   category|      date|running_sum|
|----------|------------|------|-----------|----------|-----------|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|      267.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|      867.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|      867.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|      867.0|
  

so, in order to fix this one we have to use frame, lets see below


In [88]:
// window defination
val winDef = Window
    .orderBy(col("price").asc)
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

// applying in df
df.select(col("*"), sum(col("price")).over(winDef).alias("running_sum")).show()

+----------+------------+------+-----------+----------+-----------+
|product_id|product_name| price|   category|      Date|running_sum|
+----------+------------+------+-----------+----------+-----------+
|         7|    Notebook|   5.0| Stationery|2026-02-26|        5.0|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|       17.0|
|         2|       Mouse|  25.0|Electronics|2026-02-25|       42.0|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|       72.0|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|      117.0|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|      267.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|      467.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|      667.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|      867.0|
|         1|      Laptop|1200.0|Electronics|2026-02-25|     2067.0|
+----------+------------+------+-----------+----------+-----------+



winDef: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@6039279b

In [90]:
// Rolling sum of the last 2 rows including the current row based on date

df.select(
    col("*"),
    sum(col("price"))
    .over(Window
        .orderBy(col("Date").asc)
        .rowsBetween(-2, Window.currentRow))
    .alias("runningSumLast2")).show()

+----------+------------+------+-----------+----------+---------------+
|product_id|product_name| price|   category|      Date|runningSumLast2|
+----------+------------+------+-----------+----------+---------------+
|         1|      Laptop|1200.0|Electronics|2026-02-25|         1200.0|
|         2|       Mouse|  25.0|Electronics|2026-02-25|         1225.0|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|         1270.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|          270.0|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|          395.0|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|          380.0|
|         7|    Notebook|   5.0| Stationery|2026-02-26|          185.0|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|           47.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|          217.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|          412.0|
+----------+------------+------+-----------+----------+---------

In [94]:
// Sum of the previous 2 rows (excluding the current row)

// window definations
val windowDef = Window.orderBy(col("date").asc).rowsBetween(-2, -1)

// implementation of window function in df
df.select(col("*"), sum(col("price")).over(windowDef).alias("rollingSum")).show()

+----------+------------+------+-----------+----------+----------+
|product_id|product_name| price|   category|      Date|rollingSum|
+----------+------------+------+-----------+----------+----------+
|         1|      Laptop|1200.0|Electronics|2026-02-25|      NULL|
|         2|       Mouse|  25.0|Electronics|2026-02-25|    1200.0|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|    1225.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|      70.0|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|     245.0|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|     350.0|
|         7|    Notebook|   5.0| Stationery|2026-02-26|     180.0|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|      35.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|      17.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|     212.0|
+----------+------------+------+-----------+----------+----------+



windowDef: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@61a954d0

In [95]:
// Running sum from all previous rows up to the current row based on product id

// window defination
val windowDef1 = Window.orderBy(col("product_id").asc).rowsBetween(Window.unboundedPreceding, Window.currentRow)

// implementation of window in df
df.select(col("*"), sum(col("price")).over(windowDef1).alias("runningSum")).show()

+----------+------------+------+-----------+----------+----------+
|product_id|product_name| price|   category|      Date|runningSum|
+----------+------------+------+-----------+----------+----------+
|         1|      Laptop|1200.0|Electronics|2026-02-25|    1200.0|
|         2|       Mouse|  25.0|Electronics|2026-02-25|    1225.0|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|    1270.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|    1470.0|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|    1620.0|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|    1650.0|
|         7|    Notebook|   5.0| Stationery|2026-02-26|    1655.0|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|    1667.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|    1867.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|    2067.0|
+----------+------------+------+-----------+----------+----------+



windowDef1: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@93d5795

In [96]:
// Simulating LAG without using the LAG function (previous row value)
// window defination
val windowDef1 = Window.orderBy(col("product_id").asc).rowsBetween(-1, -1)

// implementation of window in df
df.select(col("*"), sum(col("price")).over(windowDef1).alias("runningSum")).show()

+----------+------------+------+-----------+----------+----------+
|product_id|product_name| price|   category|      Date|runningSum|
+----------+------------+------+-----------+----------+----------+
|         1|      Laptop|1200.0|Electronics|2026-02-25|      NULL|
|         2|       Mouse|  25.0|Electronics|2026-02-25|    1200.0|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|      25.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|      45.0|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|     200.0|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|     150.0|
|         7|    Notebook|   5.0| Stationery|2026-02-26|      30.0|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|       5.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|      12.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|     200.0|
+----------+------------+------+-----------+----------+----------+



windowDef1: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@3d726ede

In [97]:
// Simulating LEAD without using the LEAD function (next row value)
// window defination
val windowDef1 = Window.orderBy(col("product_id").asc).rowsBetween(1, 1)

// implementation of window in df
df.select(col("*"), sum(col("price")).over(windowDef1).alias("runningSum")).show()

+----------+------------+------+-----------+----------+----------+
|product_id|product_name| price|   category|      Date|runningSum|
+----------+------------+------+-----------+----------+----------+
|         1|      Laptop|1200.0|Electronics|2026-02-25|      25.0|
|         2|       Mouse|  25.0|Electronics|2026-02-25|      45.0|
|         3|    Keyboard|  45.0|Electronics|2026-02-25|     200.0|
|         4|     Monitor| 200.0|Electronics|2026-02-25|     150.0|
|         5|  Desk Chair| 150.0|  Furniture|2026-02-25|      30.0|
|         6|   Desk Lamp|  30.0|  Furniture|2026-02-26|       5.0|
|         7|    Notebook|   5.0| Stationery|2026-02-26|      12.0|
|         8|     Pen Set|  12.0| Stationery|2026-02-27|     200.0|
|         9|     Charger| 200.0|Electronics|2026-02-27|     200.0|
|        10|        Cell| 200.0|Electronics|2026-02-28|      NULL|
+----------+------------+------+-----------+----------+----------+



windowDef1: org.apache.spark.sql.expressions.WindowSpec = org.apache.spark.sql.expressions.WindowSpec@420da1e1